# AI-Hub SOC 균열 데이터로 YOLO 학습 (Colab)

⚠️ **중요**: AI-Hub는 해외(Colab) IP의 직접 다운로드를 막습니다.
그래서 **다운로드+변환은 한국에 있는 내 PC에서** 하고,
여기 Colab에서는 **드라이브에 올린 변환 데이터로 학습만** 합니다.

## 사전 작업 (내 PC에서, 한 번)
1. PowerShell에서 실행 (data-tools 폴더):
   - `aihub_key.txt` 에 API 키 붙여넣기
   - `powershell -ExecutionPolicy Bypass -File download_convert.ps1 -FileKeys "521297,521308" -Limit 3000`
2. 만들어진 `data-tools/yolo_dataset.zip` 을 **구글 드라이브 최상위(My Drive)**에 업로드.

## 이 노트북 (Colab)
런타임 → 런타임 유형 변경 → **T4 GPU** 설정 후 위에서부터 실행.

## 1단계. GPU 확인 + 드라이브 연결

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

## 2단계. 드라이브의 데이터 압축 풀기
`yolo_dataset.zip` 을 My Drive 최상위에 올렸다고 가정합니다. (경로가 다르면 ZIP_PATH 수정)

In [ ]:
ZIP_PATH = '/content/drive/MyDrive/yolo_dataset.zip'

import os, zipfile
assert os.path.exists(ZIP_PATH), f'파일을 찾을 수 없음: {ZIP_PATH} (드라이브 업로드 위치 확인)'
os.makedirs('/content/yolo_dataset', exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/yolo_dataset')
print('압축 해제 완료')

import glob
print('train 이미지:', len(glob.glob('/content/yolo_dataset/images/train/*')))
print('val 이미지:', len(glob.glob('/content/yolo_dataset/images/val/*')))
!cat /content/yolo_dataset/data.yaml

## 3단계. data.yaml 경로 보정
변환은 내 PC에서 했으므로 data.yaml의 path를 Colab 경로로 바꿔줍니다.

In [ ]:
yaml_path = '/content/yolo_dataset/data.yaml'
lines = open(yaml_path, encoding='utf-8').read().splitlines()
out = []
for ln in lines:
    if ln.startswith('path:'):
        out.append('path: /content/yolo_dataset')
    else:
        out.append(ln)
open(yaml_path, 'w', encoding='utf-8').write('\n'.join(out) + '\n')
!cat /content/yolo_dataset/data.yaml

## 4단계. YOLO 학습

In [ ]:
%pip install -q ultralytics
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
results = model.train(
    data='/content/yolo_dataset/data.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    name='aihub_crack'
)

## 5단계. 결과 확인 + best.pt 저장
best.pt를 드라이브에 저장 + 다운로드. 이 파일을 `ai-server/models/best.pt` 에 넣으면 완성!

In [ ]:
import os, shutil
from IPython.display import Image, display

run_dir = results.save_dir
res_png = os.path.join(run_dir, 'results.png')
if os.path.exists(res_png):
    display(Image(filename=res_png, width=900))

best = os.path.join(run_dir, 'weights', 'best.pt')
shutil.copy(best, '/content/drive/MyDrive/aihub_crack_best.pt')
print('드라이브에 저장됨: /content/drive/MyDrive/aihub_crack_best.pt')

from google.colab import files
files.download(best)

## 6단계. 빠른 추론 테스트 (선택)
검증 이미지 몇 장에 탐지 결과를 그려봅니다.

In [ ]:
import glob, os
best_model = YOLO(best)
imgs = glob.glob('/content/yolo_dataset/images/val/*')[:3]
pred = best_model.predict(imgs, save=True, conf=0.25)
for p in pred:
    out_path = os.path.join(p.save_dir, os.path.basename(p.path))
    display(Image(filename=out_path, width=600))

## 데이터 늘리기
성능이 부족하면 내 PC에서 더 많은 filekey로 다시 변환:
`download_convert.ps1 -FileKeys "521297,521308,521294,521305,521298,521309,521300,521311" -Limit 10000`
→ 새 `yolo_dataset.zip` 을 드라이브에 올리고 이 노트북 재실행.